# SkillGraph Week 2 — Module 8: Knowledge Graph Preparation & Explainable Skill Gap Analysis

**Project:** SkillGraph — An Explainable Knowledge Graph-Based Framework for Career-Oriented Skill Gap Analysis and Learning Path Recommendation  
**Notebook:** `06_knowledge_graph.ipynb`  
**Module:** `scripts/build_graph.py` (`SkillGraphBuilder`)

---

## 1. Research Objective & Graph Semantics

This notebook constructs and validates the structured graph projection connecting:
- **Labor-Market Demand Entities:** `JobPosting` (2,000), `Occupation` (53), `Industry` (15), `Country` (20), `Skill` (230), `SkillCategory` (18).
- **Provenance-Rich Extraction Edges:** `REQUIRES_SKILL` and `PREFERS_SKILL` produced live by `SpacySkillExtractor(section_filter={'required', 'preferred'}, use_fast_tokenizer=True)` with exact character offsets, match methods, and confidence scores.
- **Aggregated Market & Statistical Edges:** `OCCUPATION_DEMANDS_SKILL` and `CO_OCCURS_WITH` (with Jaccard similarity and Pointwise Mutual Information `PMI`).

> **Methodological Safeguard (`CO_OCCURS_WITH` vs. `PREREQUISITE_OF`):**  
> Symmetric `CO_OCCURS_WITH` edges represent statistical co-occurrence inside job postings (`relationship_semantics = "statistical_co_occurrence_not_prerequisite"`). Co-occurrence **must not** be conflated with pedagogical prerequisite ordering, which belongs to curated curriculum graphs.

In [1]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent if NOTEBOOK_DIR.name == 'notebooks' else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.build_graph import SkillGraphBuilder, CURRICULUM_EXTENSION_SCHEMAS

builder = SkillGraphBuilder(include_broad_country_mappings=False, min_co_occurrence=5)
summary = builder.build()
validation = builder.validate_graph()

print(f"Total Active Nodes: {summary['total_active_nodes']:,}")
print(f"Total Active Edges: {summary['total_active_edges']:,}")
print(f"Referential Integrity Passed: {validation['referential_integrity_passed']} (Anomalies: {validation['total_anomalies']})")
print(f"Optional NetworkX installed in environment: {summary['networkx_installed']}")

display(pd.DataFrame([
    {'Entity / Node Type': k, 'Count': v} for k, v in summary['node_counts'].items()
]))
display(pd.DataFrame([
    {'Relationship / Edge Type': k, 'Count': v} for k, v in summary['edge_counts'].items()
]))

Total Active Nodes: 2,336
Total Active Edges: 31,503
Referential Integrity Passed: True (Anomalies: 0)
Optional NetworkX installed in environment: False


,Entity / Node Type,Count
0,Skill,230
1,SkillCategory,18
2,Occupation,53
3,Industry,15
4,Country,20
5,JobPosting,2000


,Relationship / Edge Type,Count
0,REQUIRES_SKILL,15819
1,PREFERS_SKILL,5769
2,BELONGS_TO_CATEGORY,230
3,IN_OCCUPATION,2000
4,IN_INDUSTRY,53
5,LOCATED_IN,2000
6,OCCUPATION_DEMANDS_SKILL,1645
7,CO_OCCURS_WITH,3987


## 2. Degree Centrality, Cross-Occupation Reach, and Statistical Co-Occurrence

We inspect which canonical skills exhibit the broadest cross-occupation reach across the 53 occupations and inspect the strongest positive PMI co-occurrence pairs.

In [2]:
df_occ_skill = builder.edges['OCCUPATION_DEMANDS_SKILL']
df_bridge = (
    df_occ_skill.groupby(['skill_id', 'canonical_skill'])
    .agg(
        occupations_demanding=('occupation_id', 'nunique'),
        total_job_mentions=('job_count', 'sum'),
        mean_demand_share=('demand_share', 'mean'),
    )
    .reset_index()
    .sort_values(['occupations_demanding', 'total_job_mentions'], ascending=False)
)
df_bridge['mean_demand_share'] = df_bridge['mean_demand_share'].round(4)
print('Top 10 Cross-Occupation Bridge Skills:')
display(df_bridge.head(10))

df_co = builder.edges['CO_OCCURS_WITH']
print('\nTop 10 Symmetric Statistical Co-Occurrences Ranked by PMI (count >= 25):')
display(df_co[df_co['co_occurrence_count'] >= 25].sort_values('pmi_score', ascending=False).head(10))

Top 10 Cross-Occupation Bridge Skills:


,skill_id,canonical_skill,occupations_demanding,total_job_mentions,mean_demand_share
211,SG-SK-0213,English Language,53,1881,0.9421
192,SG-SK-0194,Mentoring,53,348,0.1690
225,SG-SK-0227,Portuguese Language,50,100,0.0530
205,SG-SK-0207,Stakeholder Management,49,286,0.1668
191,SG-SK-0193,Team Leadership,48,445,0.2366
219,SG-SK-0221,Japanese Language,48,100,0.0549
220,SG-SK-0222,Korean Language,48,100,0.0554
61,SG-SK-0063,Budget Management,47,360,0.1960
194,SG-SK-0196,Decision Making,47,231,0.1318
222,SG-SK-0224,French Language,47,126,0.0715



Top 10 Symmetric Statistical Co-Occurrences Ranked by PMI (count >= 25):


,source_skill_id,source_skill,target_skill_id,target_skill,co_occurrence_count,jaccard_similarity,pmi_score,is_symmetric,relationship_semantics
824,SG-SK-0100,Pharmaceutical Dispensing,SG-SK-0101,Pharmacology,33,1.0000,5.9214,True,statistical_co_occurrence_not_prerequisite
823,SG-SK-0094,Patient Care,SG-SK-0097,Medication Administration,33,1.0000,5.9214,True,statistical_co_occurrence_not_prerequisite
1285,SG-SK-0167,Video Production,SG-SK-0168,Storyboarding,26,0.6047,5.5395,True,statistical_co_occurrence_not_prerequisite
516,SG-SK-0145,E-commerce Platform Management,SG-SK-0146,Product Listing Optimisation,43,1.0000,5.5395,True,statistical_co_occurrence_not_prerequisite
1185,SG-SK-0076,Structural Analysis,SG-SK-0201,Primavera P6,27,0.6279,5.5395,True,statistical_co_occurrence_not_prerequisite
1286,SG-SK-0174,Lesson Planning,SG-SK-0183,Teaching Certification,26,0.5909,5.5064,True,statistical_co_occurrence_not_prerequisite
864,SG-SK-0182,Academic Advising,SG-SK-0205,Program Coordination,32,0.7273,5.5064,True,statistical_co_occurrence_not_prerequisite
926,SG-SK-0050,Network Configuration,SG-SK-0051,Cisco Networking,31,0.7045,5.5064,True,statistical_co_occurrence_not_prerequisite
485,SG-SK-0126,Supply Chain Planning,SG-SK-0127,Demand Forecasting,44,1.0000,5.5064,True,statistical_co_occurrence_not_prerequisite
486,SG-SK-0163,Content Writing,SG-SK-0165,Editing and Proofreading,44,1.0000,5.5064,True,statistical_co_occurrence_not_prerequisite


## 3. Explainable Career Skill-Gap Simulation & Curriculum Schema Extension

Using `builder.explain_skill_gap(...)`, we demonstrate traceable gap recommendations where every missing competency cites empirical occupation demand share (`OCCUPATION_DEMANDS_SKILL`) and statistical co-occurrence bridges (`CO_OCCURS_WITH`) with known student skills. Curriculum entities (`UniversityMajor`, `UniversityCourse`, `Student`) remain schema-defined without synthetic course fabrication.

In [3]:
gap_ds = builder.explain_skill_gap(
    known_skills=['Python', 'SQL', 'Data Analysis', 'Communication'],
    target_occupation='Data Scientist',
    top_k=7,
)
print('Explainable Skill Gap: Student={Python, SQL, Data Analysis, Communication} -> Data Scientist')
display(gap_ds)

print('\nPlanned Curriculum & Student Schema Definitions (No Fabricated Records):')
display(pd.DataFrame([
    {'Entity': k, 'Primary Key': v['primary_key'], 'Columns': ', '.join(v['columns']), 'Status': v['status']}
    for k, v in CURRICULUM_EXTENSION_SCHEMAS.items()
]))

Explainable Skill Gap: Student={Python, SQL, Data Analysis, Communication} -> Data Scientist


,missing_skill_id,missing_skill,demand_share,required_postings,preferred_postings,co_occurring_known_skills,explanation
0,SG-SK-0027,Machine Learning,1.0000,17,0,"[Communication, Python, SQL]",Demanded in 17/17 (100.0%) postings for Data S...
1,SG-SK-0024,Statistics,1.0000,17,0,"[Communication, Data Analysis, Python, SQL]",Demanded in 17/17 (100.0%) postings for Data S...
2,SG-SK-0213,English Language,1.0000,17,0,"[Communication, Data Analysis, Python, SQL]",Demanded in 17/17 (100.0%) postings for Data S...
3,SG-SK-0028,Feature Engineering,0.5882,10,0,[Python],Demanded in 10/17 (58.8%) postings for Data Sc...
4,SG-SK-0029,Model Deployment,0.5294,0,9,[Python],Demanded in 9/17 (52.9%) postings for Data Sci...
5,SG-SK-0031,Natural Language Processing,0.4706,0,8,"[Communication, Python]",Demanded in 8/17 (47.1%) postings for Data Sci...
6,SG-SK-0012,Apache Spark,0.4706,0,8,"[Communication, Python, SQL]",Demanded in 8/17 (47.1%) postings for Data Sci...



Planned Curriculum & Student Schema Definitions (No Fabricated Records):


,Entity,Primary Key,Columns,Status
0,UniversityMajor,major_id,"major_id, major_name, institution_name, degree...",schema_ready_awaiting_curriculum_ingestion
1,UniversityCourse,course_id,"course_id, course_code, course_title, major_id...",schema_ready_awaiting_curriculum_ingestion
2,Student,student_id,"student_id, major_id, target_occupation_id, ma...",schema_ready_awaiting_student_profiles
